# TTD Price Assign — compute per-segment PoM from a delta policy

Given a universe of segments (e.g. `RankedByRevenue_March.csv`), a list of
**delta rules** (wildcard filters + a revenue policy + a delta), and an
optional **holdout list**, this notebook resolves one final price per segment
and writes it out in the format downstream systems expect.

**Output columns**: `Third_Party_Data_Provider_Id`, `Third_Party_Data_Provider_Element_Id`,
`Full_Path`, `PoM`, `delta`, `assign_date`.

## Design notes

- **Current price** comes from a second input file, `current_price.csv`
  (columns: `Third_Party_Data_Provider_Id`, `Third_Party_Data_Provider_Element_Id`,
  and a price column named `PoM`/`current_price`/`price`), joined onto the
  segment universe by provider + element id. A segment absent from that file
  falls back to `base_PoM_default[Third_Party_Data_Provider_Id]`. No prior
  notebook in this project has ever had a per-segment current-price source
  (`RankedByRevenue_March.csv` / `Price_Expt_Assignments2.csv` don't carry
  one), so a starter `current_price.csv` was generated reflecting the only
  price state actually known today: every segment at its provider's
  `base_PoM_default`. Replace it with real current-price data as soon as
  that's available — the join/fallback logic doesn't change either way.
- **Rule filter values can be a single wildcard OR a list of wildcards**,
  e.g. `'Third_Party_Data_Provider_Id': ['sharethis', 'dav2shreths']` matches
  either provider (OR, within that one rule) — on top of the existing OR
  across the whole `delta_rules` list (a segment gets a rule's delta if it
  matches *any* rule in the list). `revenue` accepts a list of policies the
  same way (e.g. `['<10', 'top:5']` keeps tiny segments AND the top 5).
- **Rule matching**: rules are evaluated independently (not in priority
  order) and a segment's final delta is the delta of whichever rule(s) match
  it. If a segment matches **more than one rule with different deltas**,
  this raises an error listing the conflicting segments/rules rather than
  silently picking one — narrow or reorder your rules' filters to resolve
  it. Matching the same delta from multiple rules is fine (not a conflict).
- **Unmatched segments**: a segment matching zero rules is treated the same
  as an explicit holdout — `delta=0`, still present in the output. Every
  segment in the input universe always gets exactly one output row.
- **Holdout always wins**: a segment in `holdout_segments` gets `delta=0`
  even if a rule would otherwise have matched it.

## Config — inputs you'll typically change

In [1]:
import pandas as pd
import numpy as np
import re

# ── current-price fallback (used where a segment is absent from current_price_file) ─
base_PoM_default = {'dav2shreths': 0.16, 'sharethis': 0.15}

# ── segment universe file ─────────────────────────────────────────────────────
data_path = "/Users/ravirajan/code/data"
output_path = "/Users/ravirajan/code/outputs/TTD"

segment_file = f"{data_path}/RankedByRevenueJuly.csv"   # e.g. RankedByRevenue.csv

# ── base price file (INPUT VARIABLE) ───────────────────────────────────────────
# Canonical, structural geo_rank/group reference (see base_price.csv design --
# rank/group are stable identifiers tied to the original experiment structure,
# not recomputed per round the way this round's own `revenue` is.
base_price_file = f"{output_path}/base_price.csv"

# ── reference file: the previous assignment (INPUT VARIABLE) ──────────────────
# Used both as this round's current-price source (it has a PoM column) AND to
# identify segments already changed by that prior assignment (see below).
reference_file = f"{output_path}/Price_Assign_2026-05-26.csv"

# ── manual holdout file (INPUT VARIABLE) ───────────────────────────────────────
# Full_Path values that must not change this round, on top of segments already
# changed by the previous assignment (computed below).
holdout_file = f"{output_path}/holdout_list_2026-07-24.csv"

# ── current price file (INPUT VARIABLE) ───────────────────────────────────────
# Columns: Third_Party_Data_Provider_Id, Third_Party_Data_Provider_Element_Id,
# and a price column named PoM / current_price / price.
current_price_file = reference_file

# ── assignment date (INPUT VARIABLE) ───────────────────────────────────────────
assign_date = "2026-07-27"

# ── campaign week-start this assignment applies to (INPUT VARIABLE) ────────────
week_start = "2026-07-27"


# ── output file (set to None to skip writing) ────────────────────────────────
output_file = f"{output_path}/Price_Assign_{assign_date}.csv"

In [2]:
def get_geo_from_path(x):
    """Same convention as TTD3_Pricing.ipynb / TTD_Pricing_Simple_Experiment.ipynb."""
    litems = x.replace(" ", "").split(">")
    check = ['Asia', 'US', 'Global', 'APAC']
    for c in check:
        if c in litems:
            return c
    return 'Unknown'

segments = pd.read_csv(segment_file)
segments['geo'] = segments['Full_Path'].apply(get_geo_from_path)

# geo_rank/group are stable identifiers joined from base_price_file (not
# recomputed from this round's own revenue) -- see design notes above.
base_price_df = pd.read_csv(base_price_file)
segments = segments.merge(
    base_price_df[['Third_Party_Data_Provider_Id', 'Full_Path', 'geo_rank', 'group']],
    on=['Third_Party_Data_Provider_Id', 'Full_Path'], how='left'
)
n_missing_rank = segments['geo_rank'].isna().sum()
if n_missing_rank:
    print(f"NOTE: {n_missing_rank} segment(s) not found in base_price_file -- "
          f"geo_rank/group set to -1 (sentinel for 'not found') so the column stays plain int")
# sentinel -1 for segments absent from base_price_file (not a real rank/group) --
# avoids leaving blanks, so this reads back as plain int in any tool, not just
# pandas with nullable Int64 dtype requested.
segments['geo_rank'] = segments['geo_rank'].fillna(-1).astype(int)
segments['group'] = segments['group'].fillna(-1).astype(int)

print(f"{len(segments):,} segments loaded from {segment_file}")
print(segments.groupby(['Third_Party_Data_Provider_Id', 'geo']).size().unstack(fill_value=0))

reference_df = pd.read_csv(reference_file)
print(f"{len(reference_df):,} rows loaded from reference_file (prior assignment)")

NOTE: 9 segment(s) not found in base_price_file -- geo_rank/group set to -1 (sentinel for 'not found') so the column stays plain int
5,744 segments loaded from /Users/ravirajan/code/data/RankedByRevenueJuly.csv
geo                           APAC  Asia  Global   US  Unknown
Third_Party_Data_Provider_Id                                  
dav2shreths                      0     2    2143    0        1
sharethis                       12  1218    1020  831      517
5,737 rows loaded from reference_file (prior assignment)


## Holdout list (INPUT VARIABLE)

Full_Path values (exact match) whose price must NOT change, regardless of
what any delta rule below says. Leave empty for no holdout.

In [3]:
# Base holdout: segments already changed by the PREVIOUS assignment
# (reference_file) shouldn't get touched again by the new bulk rule below --
# they keep whatever price that prior assignment gave them. Segments on
# restore_segments (see below) are removed from this set, since those need
# their own nonzero "restore" delta rather than being frozen at 0.
previously_changed_segments = reference_df.loc[
    (reference_df['Third_Party_Data_Provider_Id'] == 'sharethis') & (reference_df['delta'] != 0),
    'Full_Path'
].tolist()
print(f"{len(previously_changed_segments):,} segment(s) previously changed by the prior assignment")

# Manually-specified holdout, loaded from a file this round. That file
# carries its own 'comment' column (e.g. a stability-analysis note) --
# preserved per-segment so it can be appended to this round's comment
# rather than discarded.
manual_holdout_df = pd.read_csv(holdout_file)
manual_holdout_segments = manual_holdout_df['Full_Path'].tolist()
manual_holdout_labels = manual_holdout_df.set_index('Full_Path')['comment']
print(f"{len(manual_holdout_segments):,} segment(s) in the manual holdout file ({holdout_file})")

holdout_segments = list(dict.fromkeys(previously_changed_segments + manual_holdout_segments))
print(f"{len(holdout_segments):,} segment(s) held out in total")

638 segment(s) previously changed by the prior assignment
282 segment(s) in the manual holdout file (/Users/ravirajan/code/outputs/TTD/holdout_list_2026-07-24.csv)
919 segment(s) held out in total


## Delta rules (INPUT VARIABLE)

A `delta_rules` entry is a dict:
```
{'Third_Party_Data_Provider_Id': <wildcard>, 'geo': <wildcard>, 'Full_Path': <wildcard>,
 'revenue': <policy>, 'delta': <float>}
```

- **Any column in `segments`** can be used as a filter key (not just the 3
  shown above) -- whatever key you give must match an actual column name, or
  the notebook raises a clear error rather than silently matching nothing.
- **Wildcards** use SQL `LIKE` syntax: `%` matches any sequence of characters,
  `_` matches exactly one character. `'%'` alone (or omitting a key entirely)
  matches everything for that column.
- **A filter value can be a single wildcard OR a list of wildcards** --
  `'Third_Party_Data_Provider_Id': ['sharethis', 'dav2shreths']` matches
  either provider (OR'd together within that one rule). This is on top of
  the OR that already exists across the whole `delta_rules` list itself --
  a segment gets a rule's delta if it matches *any* rule in the list, and
  within one rule's field it matches if it hits *any* pattern you give.
- **`revenue`** is a policy string (or a callable, see below), not a literal
  value -- it's evaluated *after* the other filters narrow the segment set,
  so `'top:50'` means "top 50 by revenue among segments already matching
  provider/geo/Full_Path", not top 50 of the whole universe.
- **`delta`** is required on every rule.

### Revenue policy syntax

| policy | meaning |
|---|---|
| `'*'`, `'all'`, or omitted | no revenue filter |
| `'>1000'`, `'>=1000'`, `'<100'`, `'<=100'`, `'==500'`, `'!=0'` | simple comparison |
| `'between:100,1000'` | inclusive range |
| `'top:50'` | top 50 rows by revenue (within the already-filtered subset) |
| `'bottom:50'` | bottom 50 rows by revenue (within the already-filtered subset) |
| `'toppct:10'` | top 10% by revenue, i.e. revenue >= the 90th percentile (within subset) |
| a callable `fn(sub_df) -> bool Series or filtered df` | full custom logic |
| a **list** of any of the above, e.g. `['<10', 'top:5']` | OR'd together -- keeps rows matching *any* listed policy |

### Sample usage

In [4]:
delta_rules = [
    # 1. Decrease price 5% for Global sharethis segments earning > $1000/month
    {'Third_Party_Data_Provider_Id': 'sharethis', 'geo': 'Global', 'Full_Path': '%',
     'revenue': '>1000', 'delta': -0.05},

    # 2. Increase price 5% for the top 50 dav2shreths segments by revenue, Global geo
    #    (scoped to a DIFFERENT provider than rule 1, so no overlap is possible)
    {'Third_Party_Data_Provider_Id': 'dav2shreths', 'geo': 'Global', 'Full_Path': '%',
     'revenue': 'top:50', 'delta': 0.05},

    # 3. Target a taxonomy branch via wildcard, no revenue filter at all
    #    (scoped to US geo -- disjoint from rules 1/2's Global geo, so no overlap)
    {'Third_Party_Data_Provider_Id': 'sharethis', 'geo': 'US',
     'Full_Path': 'US > shopping%', 'delta': -0.03},

    # 4. Any provider, segments in a specific monthly-revenue band
    #    (scoped to Asia geo -- disjoint from rules 1/2/3, so no overlap regardless
    #    of provider/Full_Path)
    {'geo': 'Asia', 'Full_Path': '%', 'revenue': 'between:100,1000', 'delta': -0.02},

    # 5. OR-list example: a filter value can be a list, OR'd together within
    #    that one field -- matches either provider AND either taxonomy branch
    #    (scoped to APAC geo -- disjoint from rules 1-4, so no overlap)
    {'Third_Party_Data_Provider_Id': ['sharethis', 'dav2shreths'], 'geo': 'APAC',
     'Full_Path': ['%Age%', '%Gender%'], 'delta': -0.04},

    # 6. Custom revenue policy via a lambda (equivalent to example 4's revenue
    #    condition, shown for reference if you need logic the string DSL can't express)
    # {'geo': 'Asia', 'Full_Path': '%', 'revenue': lambda sub: sub['revenue'].between(100, 1000), 'delta': -0.02},

    # 7. revenue OR-list example: keep tiny segments AND the top 5 by revenue
    # {'geo': 'US', 'Full_Path': '%', 'revenue': ['<10', 'top:5'], 'delta': -0.01},

    # 8. Catch-all: everything else not already covered above gets a small
    #    decrease. Put broad/catch-all rules LAST -- if this also matched
    #    segments rules 1-5 already touched, it would need to agree on the
    #    same delta for them or the notebook raises a conflict error.
    # {'Full_Path': '%', 'revenue': '*', 'delta': -0.01},
]

### Reconstruct real experiment deltas (optional override)

To generate actual price changes matching a *known* per-segment assignment
(e.g. `Price_Expt_Assignments2.csv`, the real pricing experiment analyzed in
`TTD3_Pricing.ipynb`) instead of the wildcard-based sample rules above, set
`use_real_experiment_deltas = True` below. This builds one **exact-match**
rule per nonzero-delta segment in the experiment file -- segments in that
file with `delta==0` (controls), or segments not in the experiment at all,
don't need a rule; they get `delta=0` from the notebook's normal
unmatched-segment default, same as everywhere else.

In [5]:
use_real_experiment_deltas = False  # this run uses the bulk+restore rules below instead
experiment_file = f"{data_path}/Price_Expt_Assignments2.csv"

def build_delta_rules_from_experiment(experiment_file):
    exp = pd.read_csv(experiment_file)
    exp['delta'] = exp['assignment'] * exp['incr']
    nz = exp[exp['delta'] != 0]
    dupes = nz[nz.duplicated(subset=['Third_Party_Data_Provider_Id', 'Full_Path'], keep=False)]
    if len(dupes) > 0:
        raise ValueError(
            f"{experiment_file} has {len(dupes)} row(s) with duplicate "
            f"(Third_Party_Data_Provider_Id, Full_Path) -- can't build exact-match "
            f"rules unambiguously from it:\n{dupes[['Third_Party_Data_Provider_Id', 'Full_Path', 'delta']].to_string(index=False)}"
        )
    return [
        {'Third_Party_Data_Provider_Id': row['Third_Party_Data_Provider_Id'],
         'Full_Path': row['Full_Path'], 'delta': row['delta']}
        for _, row in nz.iterrows()
    ]

if use_real_experiment_deltas:
    delta_rules = build_delta_rules_from_experiment(experiment_file)
    print(f"delta_rules overridden: {len(delta_rules):,} exact-match rules reconstructed from {experiment_file}")

### This run's actual policy: bulk decrease + restore list

Two pieces, combined into `delta_rules`:

1. **Bulk rule**: `-0.05` for all `sharethis` segments with `revenue <= 1000`,
   excluding segments already changed by the previous assignment
   (`holdout_segments`, computed above) and excluding `restore_segments`
   (below) — computed as an explicit `Full_Path` list (not a wildcard +
   revenue-policy rule) so there's no possibility of it overlapping with
   the restore rules or holdout by construction, rather than relying on
   rule-priority (this engine has none -- see design notes at the top).
2. **Restore rules**: for `restore_segments`, bring the price back to
   `base_PoM_default[provider]` — computed *per segment* from its own
   current price in `reference_df` (not a fixed delta, since different
   segments moved by different amounts in the prior assignment).

In [6]:
restore_segments = [
    'Global > Validated Demographic > Age > 18-44',
    'Global > people & society > social issues & advocacy > green living & environmental issues',
    'Global > hobbies & leisure > outdoors',
    'Global > travel > tourist destinations > mountain & ski resorts',
    'Global > business & industrial > hospitality industry > food service',
    'Global > law & government > military > veterans',
    'Global > computers & electronics > networking > vpn & remote access',
    'Global > finance > investing',
    'Global > Validated Demographic > Gender and Age > Females 18-34',  # corrected from "Females" -- no exact match otherwise
    'Global > business & industrial > transportation & logistics > freight & trucking',
]

# restore_segments always wins over the "already changed" holdout -- remove
# any overlap so the restore rule isn't silently zeroed out by holdout.
_overlap = set(restore_segments) & set(holdout_segments)
if _overlap:
    print(f"{len(_overlap)} restore segment(s) removed from holdout_segments so their restore rule applies:")
    for s in sorted(_overlap):
        print(" ", s)
    holdout_segments = [s for s in holdout_segments if s not in restore_segments]


def build_restore_rules(restore_segments, reference_df, base_PoM_default, price_col='PoM'):
    """One exact-match rule per restore segment, delta computed to bring its
    CURRENT price (from reference_df) back to its provider's default."""
    sub = reference_df[reference_df['Full_Path'].isin(restore_segments)].copy()
    missing = set(restore_segments) - set(sub['Full_Path'])
    if missing:
        print(f"WARNING: {len(missing)} restore segment(s) not found in reference_df: {sorted(missing)}")
    sub['_target'] = sub['Third_Party_Data_Provider_Id'].map(base_PoM_default)
    sub['_restore_delta'] = (sub['_target'] - sub[price_col]).round(6)
    return [
        {'Third_Party_Data_Provider_Id': row['Third_Party_Data_Provider_Id'],
         'Full_Path': row['Full_Path'], 'delta': row['_restore_delta']}
        for _, row in sub.iterrows()
    ]

restore_rules = build_restore_rules(restore_segments, reference_df, base_PoM_default)
print(f"\n{len(restore_rules)} restore rule(s):")
for r in restore_rules:
    print(" ", r)

_excluded = set(holdout_segments) | set(restore_segments)
_bulk_target = segments[
    (segments['Third_Party_Data_Provider_Id'] == 'sharethis') &
    (segments['revenue'] <= 1000) &
    (~segments['Full_Path'].isin(_excluded))
]['Full_Path'].tolist()
bulk_rule = {'Third_Party_Data_Provider_Id': 'sharethis', 'Full_Path': _bulk_target, 'delta': -0.05}
print(f"\nbulk rule target: {len(_bulk_target):,} segments")

delta_rules = [bulk_rule] + restore_rules
print(f"delta_rules: {len(delta_rules)} rule(s) total ({len(_bulk_target):,}-segment bulk rule + {len(restore_rules)} restore rules)")

10 restore segment(s) removed from holdout_segments so their restore rule applies:
  Global > Validated Demographic > Age > 18-44
  Global > Validated Demographic > Gender and Age > Females 18-34
  Global > business & industrial > hospitality industry > food service
  Global > business & industrial > transportation & logistics > freight & trucking
  Global > computers & electronics > networking > vpn & remote access
  Global > finance > investing
  Global > hobbies & leisure > outdoors
  Global > law & government > military > veterans
  Global > people & society > social issues & advocacy > green living & environmental issues
  Global > travel > tourist destinations > mountain & ski resorts

10 restore rule(s):
  {'Third_Party_Data_Provider_Id': 'sharethis', 'Full_Path': 'Global > people & society > social issues & advocacy > green living & environmental issues', 'delta': -0.05}
  {'Third_Party_Data_Provider_Id': 'sharethis', 'Full_Path': 'Global > Validated Demographic > Age > 18-44',

## Engine — wildcard matching, revenue policy, rule application

In [7]:
def like_to_regex(pattern):
    """SQL LIKE-style wildcard: % = any sequence, _ = any single char.
    Splits on the wildcard chars first and escapes only the literal chunks --
    escaping the whole string first (then trying to un-escape %/_) doesn't
    work, since re.escape doesn't actually escape % or _ in the first place."""
    pattern = str(pattern)
    parts = re.split(r'([%_])', pattern)
    regex_parts = []
    for part in parts:
        if part == '%':
            regex_parts.append('.*')
        elif part == '_':
            regex_parts.append('.')
        else:
            regex_parts.append(re.escape(part))
    return re.compile('^' + ''.join(regex_parts) + '$')


def _has_wildcard_chars(p):
    return isinstance(p, str) and ('%' in p or '_' in p)


def wildcard_match(series, pattern):
    """`pattern` can be a single wildcard string, or a list/tuple/set of
    wildcard strings -- matches if ANY of them match (OR). A pattern with no
    %/_ at all skips the regex entirely (vectorized equality) -- exact-match
    rules (e.g. reconstructing a known per-segment assignment) are common
    and this is meaningfully faster at scale."""
    patterns = pattern if isinstance(pattern, (list, tuple, set)) else [pattern]
    str_series = None
    mask = pd.Series(False, index=series.index)
    for p in patterns:
        if p is None or p == '%':
            return pd.Series(True, index=series.index)
        if str_series is None:
            str_series = series.astype(str)
        if _has_wildcard_chars(p):
            rx = like_to_regex(p)
            mask |= str_series.apply(lambda x: bool(rx.match(x)))
        else:
            mask |= (str_series == str(p))
    return mask


def _apply_single_revenue_policy(sub_df, policy):
    if policy is None or policy in ('*', 'all'):
        return sub_df
    if callable(policy):
        result = policy(sub_df)
        if isinstance(result, pd.Series) and result.dtype == bool:
            return sub_df[result]
        return result  # assume already filtered

    policy = policy.strip()
    m = re.match(r'^(>=|<=|==|!=|>|<)\s*(-?\d+\.?\d*)$', policy)
    if m:
        op, val = m.group(1), float(m.group(2))
        rev = sub_df['revenue']
        ops = {'>': rev.gt, '>=': rev.ge, '<': rev.lt, '<=': rev.le, '==': rev.eq, '!=': rev.ne}
        return sub_df[ops[op](val)]

    m = re.match(r'^between:\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)$', policy)
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
        return sub_df[sub_df['revenue'].between(lo, hi)]

    m = re.match(r'^top:\s*(\d+)$', policy)
    if m:
        return sub_df.nlargest(int(m.group(1)), 'revenue')

    m = re.match(r'^bottom:\s*(\d+)$', policy)
    if m:
        return sub_df.nsmallest(int(m.group(1)), 'revenue')

    m = re.match(r'^toppct:\s*(-?\d+\.?\d*)$', policy)
    if m:
        pct = float(m.group(1))
        threshold = sub_df['revenue'].quantile(1 - pct / 100)
        return sub_df[sub_df['revenue'] >= threshold]

    raise ValueError(f"Unrecognized revenue policy: {policy!r}")


def apply_revenue_policy(sub_df, policy):
    """See the revenue-policy table in the markdown above. `policy` can also
    be a list of policies, OR'd together (union of rows matching any of
    them) -- e.g. ['<10', 'top:5'] keeps tiny segments AND the top 5."""
    if isinstance(policy, (list, tuple)):
        matched_idx = pd.Index([])
        for p in policy:
            matched_idx = matched_idx.union(_apply_single_revenue_policy(sub_df, p).index)
        return sub_df.loc[sub_df.index.intersection(matched_idx)]
    return _apply_single_revenue_policy(sub_df, policy)


FILTERABLE_META_KEYS = {'delta', 'revenue'}  # not matched via wildcard directly

def apply_rule(df, rule):
    sub = df
    for key, pattern in rule.items():
        if key in FILTERABLE_META_KEYS:
            continue
        if key not in df.columns:
            raise KeyError(f"Rule references unknown column {key!r}: {rule}")
        sub = sub[wildcard_match(sub[key], pattern)]
    if 'revenue' in rule:
        sub = apply_revenue_policy(sub, rule['revenue'])
    return sub

In [8]:
import os

def assign_deltas(df, delta_rules, holdout_segments):
    """Resolve one delta per segment. Matching the same delta from multiple
    rules is fine; matching DIFFERENT deltas from multiple rules raises --
    see design notes above. Segments matching no rule (and not in the
    holdout) default to delta=0, same as an explicit holdout."""
    matches = {idx: [] for idx in df.index}
    for rule_idx, rule in enumerate(delta_rules):
        if 'delta' not in rule:
            raise KeyError(f"Rule {rule_idx} missing required 'delta' key: {rule}")
        matched = apply_rule(df, rule)
        for idx in matched.index:
            matches[idx].append((rule_idx, rule['delta']))

    resolved_delta, conflicts = {}, []
    for idx, m in matches.items():
        deltas = {d for _, d in m}
        if len(deltas) > 1:
            conflicts.append({'Full_Path': df.loc[idx, 'Full_Path'], 'matches': m})
        elif len(deltas) == 1:
            resolved_delta[idx] = deltas.pop()
        else:
            resolved_delta[idx] = 0.0

    if conflicts:
        msg = "\n".join(f"  {c['Full_Path']!r}: matched rules {c['matches']}" for c in conflicts[:20])
        more = f"\n  ... and {len(conflicts) - 20} more" if len(conflicts) > 20 else ""
        raise ValueError(
            f"{len(conflicts)} segment(s) matched multiple rules with DIFFERENT deltas -- "
            f"narrow or reorder your rules' filters to resolve:\n{msg}{more}"
        )

    delta_series = pd.Series(resolved_delta).reindex(df.index).fillna(0.0)
    holdout_mask = df['Full_Path'].isin(holdout_segments)
    n_held = int(holdout_mask.sum())
    delta_series = delta_series.where(~holdout_mask, 0.0)
    print(f"{n_held} segment(s) held out (delta forced to 0 regardless of rule matches)")
    return delta_series


JOIN_KEYS = ['Third_Party_Data_Provider_Id', 'Third_Party_Data_Provider_Element_Id']

def load_current_price(current_price_file):
    if not os.path.exists(current_price_file):
        raise FileNotFoundError(
            f"current_price_file not found: {current_price_file!r}\n"
            f"Expected a CSV with columns {JOIN_KEYS + ['PoM']} "
            f"(price column name can also be 'current_price' or 'price')."
        )
    cp = pd.read_csv(current_price_file)
    price_col = next((c for c in ['PoM', 'current_price', 'price'] if c in cp.columns), None)
    if price_col is None:
        raise KeyError(
            f"{current_price_file} has no recognized price column -- expected one of "
            f"'PoM' / 'current_price' / 'price', got columns {cp.columns.tolist()}"
        )
    missing_keys = [k for k in JOIN_KEYS if k not in cp.columns]
    if missing_keys:
        raise KeyError(f"{current_price_file} is missing join key column(s): {missing_keys}")
    dupes = cp[cp.duplicated(subset=JOIN_KEYS, keep=False)]
    if len(dupes) > 0:
        raise ValueError(
            f"{current_price_file} has {len(dupes)} row(s) with duplicate {JOIN_KEYS} keys -- "
            f"each segment must appear at most once:\n{dupes[JOIN_KEYS].head(10).to_string(index=False)}"
        )
    return cp[JOIN_KEYS + [price_col]].rename(columns={price_col: '_current_price'})


def get_current_price(df, current_price_file, base_PoM_default):
    """Per-segment current PoM: join current_price_file by
    (Third_Party_Data_Provider_Id, Third_Party_Data_Provider_Element_Id),
    falling back to base_PoM_default[provider] for any segment absent from
    that file -- see design notes above."""
    provider_default = df['Third_Party_Data_Provider_Id'].map(base_PoM_default)
    if provider_default.isna().any():
        missing = df.loc[provider_default.isna(), 'Third_Party_Data_Provider_Id'].unique()
        raise KeyError(f"base_PoM_default has no entry for provider(s): {list(missing)}")

    cp = load_current_price(current_price_file)
    merged = df.merge(cp, on=JOIN_KEYS, how='left')
    assert len(merged) == len(df), "current_price join changed row count -- check for duplicate keys"
    n_from_file = int(merged['_current_price'].notna().sum())
    print(f"current price: {n_from_file:,} segment(s) from {current_price_file}, "
          f"{len(df) - n_from_file:,} fell back to base_PoM_default")
    return merged['_current_price'].where(merged['_current_price'].notna(), provider_default.values)

## Run — resolve deltas, compute PoM, build output

In [9]:
delta_series = assign_deltas(segments, delta_rules, holdout_segments)
current_price = get_current_price(segments, current_price_file, base_PoM_default)

output = segments[['Third_Party_Data_Provider_Id', 'Third_Party_Data_Provider_Element_Id',
                    'Full_Path', 'geo', 'revenue', 'geo_rank', 'group']].copy()
output['delta'] = delta_series
output['PoM_old'] = current_price
output['PoM'] = current_price + output['delta']
output['assign_date'] = assign_date
output['week_start'] = week_start

# Comment: super-brief description of the action taken. "experiment1" refers
# to the prior assignment (reference_file / Price_Assign_2026-05-26.csv) --
# precedence below (each overwrites the previous, so restore/rollback always
# wins, matching the actual delta precedence in assign_deltas).
output['comment'] = 'no change'
output.loc[output['Full_Path'].isin(previously_changed_segments), 'comment'] = 'experiment1 retained'
output.loc[output['Full_Path'].isin(_bulk_target), 'comment'] = 'experiment1 adoption'
# manual_holdout_labels already reads 'experiment 1 holdout' -- self-
# descriptive, no need for a redundant 'holdout | ' prefix on top of it.
_holdout_mask = output['Full_Path'].isin(manual_holdout_segments)
output.loc[_holdout_mask, 'comment'] = output.loc[_holdout_mask, 'Full_Path'].map(manual_holdout_labels)
output.loc[output['Full_Path'].isin(restore_segments), 'comment'] = 'experiment1 rollback'

output = output[['Third_Party_Data_Provider_Id', 'Third_Party_Data_Provider_Element_Id',
                  'Full_Path', 'geo', 'revenue', 'geo_rank', 'group', 'PoM_old', 'PoM', 'delta',
                  'assign_date', 'week_start', 'comment']]

print(f"\n{len(output):,} segments assigned, {(output['delta'] != 0).sum():,} with a nonzero delta")
print(output['delta'].value_counts().sort_index())
print()
print(output['comment'].value_counts())
output.head(10)

913 segment(s) held out (delta forced to 0 regardless of rule matches)
current price: 5,735 segment(s) from /Users/ravirajan/code/outputs/TTD/Price_Assign_2026-05-26.csv, 9 fell back to base_PoM_default

5,744 segments assigned, 2,008 with a nonzero delta
delta
-0.05    2003
 0.00    3736
 0.05       5
Name: count, dtype: int64

comment
no change               2823
experiment1 adoption    1998
experiment1 retained     627
experiment 1 holdout     286
experiment1 rollback      10
Name: count, dtype: int64


,Third_Party_Data_Provider_Id,Third_Party_Data_Provider_Element_Id,Full_Path,geo,revenue,geo_rank,group,PoM_old,PoM,delta,assign_date,week_start,comment
0,dav2shreths,38abd508,Data Alliance > Travel > Interest > Global > T...,Global,673.99,13,4,0.16,0.16,0.0,2026-07-27,2026-07-27,no change
1,dav2shreths,98202665,Data Alliance > Business & Finance > Interest ...,Global,474.80,1,0,0.16,0.16,0.0,2026-07-27,2026-07-27,no change
2,dav2shreths,5e782346,Data Alliance > Demographic > Interest > Globa...,Global,284.34,2,0,0.16,0.16,0.0,2026-07-27,2026-07-27,no change
3,dav2shreths,95caffea,Data Alliance > Business & Finance > Interest ...,Global,255.17,3,0,0.16,0.16,0.0,2026-07-27,2026-07-27,no change
4,dav2shreths,76862030,Data Alliance > CPG > Interest > Global > Heal...,Global,247.39,7,2,0.16,0.16,0.0,2026-07-27,2026-07-27,no change
5,dav2shreths,21cba04f,Data Alliance > Demographic > Interest > Globa...,Global,209.67,4,1,0.16,0.16,0.0,2026-07-27,2026-07-27,no change
6,dav2shreths,5b7afa9a,Data Alliance > CPG > Interest > Global > Food...,Global,159.85,39,12,0.16,0.16,0.0,2026-07-27,2026-07-27,no change
7,dav2shreths,0158570f,Data Alliance > Travel > Interest > Global > D...,Global,158.47,6,1,0.16,0.16,0.0,2026-07-27,2026-07-27,no change
8,dav2shreths,89c399a8,Data Alliance > Travel > Interest > Global > A...,Global,153.02,5,1,0.16,0.16,0.0,2026-07-27,2026-07-27,no change
9,dav2shreths,61d9e70c,Data Alliance > Health & Wellness > Interest >...,Global,138.79,10,3,0.16,0.16,0.0,2026-07-27,2026-07-27,no change


## Experiment 2 overlay (optional)

Applies AFTER the main resolution above, as a targeted override for a
specific candidate list (`experiment2_candidates_file`) — first restores
each candidate to its provider's `base_PoM_default` (same computed-delta
approach as `restore_segments` above, based on its own `PoM_old` from this
run), then adds an independent random `+/- experiment2_delta_magnitude` on
top. Wins over every other rule/holdout for these specific segments, same
precedence idea as `restore_segments`.

`comment` distinguishes segments that were ever allocated a `group` in the
original experiment (`Price_Expt_Assignments2.csv`, regardless of what their
delta was there — even a `delta=0` control-arm member counts) from ones that
were never part of it at all.

In [10]:
run_experiment2 = True
experiment2_candidates_file = f"{output_path}/experiment2_candidates_2026-07-24.csv"

if run_experiment2:
    # experiment2_candidates_file carries its own 'comment' column (preserved
    # per-segment, appended to this round's comment) AND its own 'new_delta'
    # column -- the +/-0.025 (or 0) perturbation is now assigned externally
    # (e.g. by a stability-based process) rather than generated by this
    # notebook, so we use it directly instead of a random draw.
    exp2_df = pd.read_csv(experiment2_candidates_file)
    exp2_keys_set = set(zip(exp2_df['Third_Party_Data_Provider_Id'], exp2_df['Full_Path']))
    exp2_comments = dict(zip(zip(exp2_df['Third_Party_Data_Provider_Id'], exp2_df['Full_Path']), exp2_df['comment']))
    exp2_perturbation = dict(zip(zip(exp2_df['Third_Party_Data_Provider_Id'], exp2_df['Full_Path']), exp2_df['new_delta']))

    # 'was this segment REALLY deployed in experiment 1' -- reuses reference_df
    # (already loaded, = Price_Assign_2026-05-26.csv) rather than the raw
    # Price_Expt_Assignments2.csv: that file's own `group` includes dav2shreths,
    # which had planned groups but was never actually deployed -- reference_df's
    # `comment` already correctly excludes it (see the dav2shreths correction).
    exp1_grouped = set(zip(
        reference_df.loc[reference_df['comment'].isin(['experiment 1', 'control 1']), 'Third_Party_Data_Provider_Id'],
        reference_df.loc[reference_df['comment'].isin(['experiment 1', 'control 1']), 'Full_Path']
    ))

    output_keys = list(zip(output['Third_Party_Data_Provider_Id'], output['Full_Path']))
    exp2_mask = pd.Series([k in exp2_keys_set for k in output_keys], index=output.index)
    n_found = int(exp2_mask.sum())
    if n_found < len(exp2_df):
        print(f"WARNING: only {n_found} / {len(exp2_df)} experiment2 candidates found in output")
    else:
        print(f"{n_found} / {len(exp2_df)} experiment2 candidates found in output")

    base_default_series = output['Third_Party_Data_Provider_Id'].map(base_PoM_default)
    restore_delta = base_default_series - output['PoM_old']
    perturbation = pd.Series([exp2_perturbation.get(k, 0.0) for k in output_keys], index=output.index)
    new_delta = restore_delta + perturbation
    new_pom = output['PoM_old'] + new_delta

    # Only prepend the rollback note when it adds real information -- the file's
    # own comment ('experiment 2' / 'control 2') already says what this round
    # assigned; a fixed 'experiment 2' prefix would just duplicate that verbatim
    # whenever was_in_exp1 is False.
    was_in_exp1 = pd.Series([k in exp1_grouped for k in output_keys], index=output.index)
    exp2_comment_series = pd.Series([exp2_comments.get(k, '') for k in output_keys], index=output.index)
    rollback_prefix = np.where(was_in_exp1, 'experiment 1 rollback | ', '')
    new_comment = pd.Series(rollback_prefix, index=output.index) + exp2_comment_series

    output.loc[exp2_mask, 'delta'] = new_delta[exp2_mask]
    output.loc[exp2_mask, 'PoM'] = new_pom[exp2_mask]
    output.loc[exp2_mask, 'comment'] = pd.Series(new_comment, index=output.index)[exp2_mask]

    print()
    print(output.loc[exp2_mask, 'comment'].value_counts())
    print()
    print(output.loc[exp2_mask, ['Full_Path', 'PoM_old', 'PoM', 'delta', 'comment']].head(10).to_string())

329 / 328 experiment2 candidates found in output

comment
experiment 2                            161
control 2                                86
experiment 1 rollback | experiment 2     53
experiment 1 rollback | control 2        29
Name: count, dtype: int64

                                                               Full_Path  PoM_old    PoM  delta                               comment
2146                                      B2B > Industry > Food Services     0.15  0.125 -0.025                          experiment 2
2148                    US > health > nutrition > vitamins & supplements     0.15  0.150  0.000                             control 2
2152         US > Validated Demographic > Gender and Age > Females 18-34     0.15  0.175  0.025                          experiment 2
2155     Global > Validated Demographic > Gender and Age > Females 18-34     0.10  0.125  0.025  experiment 1 rollback | experiment 2
2160  US > people & society > family & relationships > Military Famil

In [11]:
# per-rule hit counts -- sanity check that each rule matched what you expected.
# With a large reconstructed rule list (one exact-match rule per segment) or
# a rule with a large Full_Path list value, print a summary instead of
# dumping every rule/list inline.
def _rule_repr(rule, max_list_items=5):
    parts = []
    for k, v in rule.items():
        if isinstance(v, (list, tuple)) and len(v) > max_list_items:
            v = f"[{len(v)} items: {v[0]!r}, {v[1]!r}, ... ]"
        parts.append(f"{k}={v!r}")
    return "{" + ", ".join(parts) + "}"

if len(delta_rules) <= 20:
    for i, rule in enumerate(delta_rules):
        matched = apply_rule(segments, rule)
        print(f"rule {i}: {_rule_repr(rule)}")
        print(f"  -> {len(matched):,} segments matched")
else:
    hit_counts = [len(apply_rule(segments, rule)) for rule in delta_rules]
    print(f"{len(delta_rules):,} rules -- hit-count distribution:")
    print(pd.Series(hit_counts).value_counts().sort_index())
    n_zero_hit = sum(1 for c in hit_counts if c == 0)
    if n_zero_hit:
        print(f"WARNING: {n_zero_hit} rule(s) matched zero segments")

rule 0: {Third_Party_Data_Provider_Id='sharethis', Full_Path="[1997 items: 'US > people & society > Language > Spanish', 'US > Validated Demographic > Gender and Age > Females 18-34', ... ]", delta=-0.05}
  -> 1,998 segments matched
rule 1: {Third_Party_Data_Provider_Id='sharethis', Full_Path='Global > people & society > social issues & advocacy > green living & environmental issues', delta=-0.05}
  -> 1 segments matched
rule 2: {Third_Party_Data_Provider_Id='sharethis', Full_Path='Global > Validated Demographic > Age > 18-44', delta=0.05}
  -> 1 segments matched
rule 3: {Third_Party_Data_Provider_Id='sharethis', Full_Path='Global > travel > tourist destinations > mountain & ski resorts', delta=0.05}
  -> 1 segments matched
rule 4: {Third_Party_Data_Provider_Id='sharethis', Full_Path='Global > hobbies & leisure > outdoors', delta=-0.05}
  -> 1 segments matched
rule 5: {Third_Party_Data_Provider_Id='sharethis', Full_Path='Global > business & industrial > hospitality industry > food serv

In [12]:
if output_file is not None:
    output.to_csv(output_file, index=False)
    print(f"wrote {len(output):,} rows to {output_file}")
else:
    print("output_file is None -- not writing to disk")

wrote 5,744 rows to /Users/ravirajan/code/outputs/TTD/Price_Assign_2026-07-27.csv
